# Notebook 1: Capture and Score

**Capture, Score, Gate, Promote, Decay.** This notebook is the first two. Capture wraps an observation in its provenance. Score describes it with three numbers. Neither one decides anything. The decision is notebook 2, and it is only as good as what these two hand it.

Most of this notebook is working code you read and run. Two cells have a TODO: one line each, marked, with a check cell after them. If you are new to this: the TODOs are plain Python, no vector math. If you have built RAG before: Score is the part that has no equivalent on the read path, so slow down there. If you run agents in production: notice what provenance we insist on and tell me what you would add.

Nothing in this notebook writes to the store. Score reads it.

In [1]:
# Make `import memory` work whether Jupyter opened the repo root or the notebooks folder.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "memory" / "__init__.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import contextlib
import io

with contextlib.redirect_stderr(io.StringIO()):  # a noisy third-party import warning we do not need in the room
    import weaviate.classes.init  # noqa: F401

from memory import capture, collection_name, connect, describe_tier, fetch_all, score_candidate

print(describe_tier())

WEAVIATE_MODE=docker (tier 1, local Docker container), collection MemoryRecord


In [2]:
client = connect()
if not client.collections.exists(collection_name()):
    raise RuntimeError("no collection yet. Run notebook 0 first; it creates and seeds the store.")
collection = client.collections.get(collection_name())  # read it; this notebook creates nothing
start_count = len(fetch_all(collection, status="active"))
if start_count == 0:
    raise RuntimeError("no active records. Run the 'fell behind' cell in notebook 0 to seed the handbook.")
if start_count != 50:
    print(
        f"store holds {start_count} records, not the 50 handbook facts notebook 0 seeds. Scoring still "
        "works; the numbers below may differ a little from the text. For a clean start run the "
        "'fell behind' cell in notebook 0, which resets the store."
    )
print(f"store holds {start_count} active records")

store holds 50 active records


## Capture: a candidate is a proposal, not a fact

An agent in a support session hears something worth keeping. In an ungoverned system that sentence goes straight into the store with a vector and a timestamp. Here it becomes a `Candidate`, and a candidate has to answer five questions before anything else happens: what was observed, who said it to the system, what kind of source that is, where it was observed, and how sure the source was.

The one that matters most is `source`. It is who said it **to the system**, not who the text claims said it. A chat message that begins "Daniel Reyes said..." has a source of whoever typed it, and that distinction is what stops the authority spoof in notebook 2.

In [3]:
cand = capture(
    text="Halvorsen Dental asked that invoices go to their accounts payable mailbox.",
    source="agent:harbor",            # the support agent that observed it
    source_type="agent",              # human | document | agent | tool | web
    context="support-session-4471",   # where it was observed
    confidence=0.8,                   # the source's own confidence, 0 to 1
    topic="customer",
    scope="customer:halvorsen-dental",
)
for k, v in cand.as_dict().items():
    print(f"{k:14s} {v}")
print("\nproblems:", cand.validate() or "none")

text           Halvorsen Dental asked that invoices go to their accounts payable mailbox.
source         agent:harbor
source_type    agent
context        support-session-4471
confidence     0.8
topic          customer
scope          customer:halvorsen-dental
tier           session
captured_at    2026-10-16T00:57:01.554335+00:00
candidate_id   ad0db5c42b22

problems: none


`validate()` checks shape, not truth. It cannot tell you whether Halvorsen Dental really said that. It can tell you when a candidate arrived without the provenance a later reviewer would need, and the gate treats that as a reason to deny. Here is a sloppy one.

In [4]:
sloppy = capture(
    text="Halvorsen Dental asked that invoices go to their accounts payable mailbox.",
    source="forum",       # no kind: prefix, so nobody can tell what sort of source this was
    source_type="agent",
    context="",           # observed nowhere in particular
    confidence=1.5,       # more than certain
)
for problem in sloppy.validate():
    print("-", problem)

- source must look like kind:name, for example user:ada or doc:handbook
- context is empty
- confidence must be a number between 0 and 1


## Score: three numbers, no decision

| Score | Question | How it is computed here |
|---|---|---|
| `novelty` | Is this new? | 1 minus the similarity of the nearest active record |
| `utility` | Is it worth keeping? | Specific, confident, right-sized facts score high; hedged fragments score low |
| `contradiction` | Does it disagree with something stored? | A close neighbor on the same topic with a flipped negation or a changed number |

Scoring runs a nearest-neighbor query against the store and then does arithmetic on the result. It never writes. It also never decides: a duplicate is not rejected here, it is described as a duplicate, and notebook 3 turns that into a corroboration instead of a second copy.

In [5]:
scores = score_candidate(collection, cand)
print(scores.summary())
print("\nnearest active records:")
for n in scores.nearest:
    print(f"  {n.similarity:.3f} [{n.tier}/{n.topic}] {n.text[:70]}")

novelty=0.48 utility=0.96 contradiction=0.00

nearest active records:
  0.517 [org/policy] Vendor invoices are paid on net 30 terms and only against a matching p
  0.491 [org/customer] Halvorsen Dental is a multi-location dental group in Seattle that buys
  0.474 [org/customer] Pinecrest Clinics orders janitorial supplies for 12 sites and prefers 
  0.419 [org/customer] Riverbend School District has a tax-exempt account and requires a purc
  0.354 [org/process] Every refund, credit, and discount is logged in Ledgerstone with the a


Novelty 0.48 means the closest thing in the store is about half similar. That neighbor is the vendor invoice policy, which is near in meaning and not in conflict. Nothing stored is the same fact, so this candidate is worth a look.

### Novelty and duplicates

Feed the store a sentence it already holds, word for word, and novelty drops to zero. `duplicate_of` points at the record. That is the signal Promote uses to corroborate an existing memory rather than store it twice.

In [6]:
dup = capture(
    text="Refunds over $500 require approval from a support manager before they are issued.",
    source="agent:harbor", source_type="agent", context="support-session-4471",
    confidence=0.8, topic="policy",
)
print(score_candidate(collection, dup).summary())

novelty=0.00 utility=0.96 contradiction=0.00 | duplicate of: 'Refunds over $500 require approval from a support '


### Utility, and TODO 1

`utility_score` rewards the shape of a useful fact: enough content words to mean something, a number or a proper noun so it is about something in particular, and a confident source. It is a heuristic and an honest one. It cannot tell a true fact from a false one. It can tell a fact from a shrug.

The version below is missing one line. Hedged sentences ("I think", "maybe", "might") should lose 0.2. `HEDGES` is the set of hedge words and `tokens` is the set of content words in the text. Finish it.

In [7]:
import re

from memory.score import HEDGES, content_tokens, numbers_in


def my_utility_score(candidate) -> float:
    text = candidate.text.strip()
    tokens = content_tokens(text)
    score = 0.3
    if 6 <= len(tokens) <= 40:
        score += 0.3  # long enough to mean something, short enough to be one fact
    if numbers_in(text) or re.search(r"\b[A-Z][a-z]+(?: [A-Z][a-z]+)+\b", text):
        score += 0.2  # names a number or a proper noun
    # TODO 1: if any hedge word appears in `tokens`, subtract 0.2
    score += 0.2 * float(candidate.confidence)
    return round(min(1.0, max(0.0, score)), 4)


hedged = capture(
    text="I think the warehouse might close early sometimes.",
    source="agent:harbor", source_type="agent", context="support-session-4471", confidence=0.8,
)
print(f"{my_utility_score(cand):.2f}  {cand.text}")
print(f"{my_utility_score(hedged):.2f}  {hedged.text}")

0.96  Halvorsen Dental asked that invoices go to their accounts payable mailbox.
0.76  I think the warehouse might close early sometimes.


### Contradiction, and TODO 2

The contradiction check asks two cheap questions of every neighbor that is close enough to be about the same thing, on the same topic, and not a duplicate: did a negation appear or disappear, and did a number change? Either one, plus enough word overlap to rule out coincidence, counts as a conflict, and the strength of the conflict is the similarity of the record it conflicts with.

The version below handles the negation question and skips the number one. `cand_nums` already holds the numbers in the candidate; `numbers_in(n.text)` gives you the neighbor's. A number has changed when both texts carry numbers and the two sets differ. Finish it.

In [8]:
from memory.score import DUPLICATE_SIMILARITY, NEGATIONS, RELATED_SIMILARITY, nearest_records
from memory.embed import embed_one


def my_contradiction_score(candidate, neighbors):
    cand_tokens = content_tokens(candidate.text)
    cand_neg = bool(cand_tokens & NEGATIONS)
    cand_nums = numbers_in(candidate.text)
    best, best_neighbor = 0.0, None
    for n in neighbors:
        if n.similarity < RELATED_SIMILARITY or n.similarity >= DUPLICATE_SIMILARITY:
            continue  # not about the same thing, or the same memory outright
        if n.topic and candidate.topic and n.topic != candidate.topic:
            continue
        other_tokens = content_tokens(n.text)
        overlap = len(cand_tokens & other_tokens) / max(1, len(cand_tokens | other_tokens))
        flipped_negation = cand_neg != bool(other_tokens & NEGATIONS)
        changed_number = False  # TODO 2: True when both texts carry numbers and they differ
        if (flipped_negation or changed_number) and overlap >= 0.2:
            if n.similarity > best:
                best, best_neighbor = round(n.similarity, 4), n
    return best, best_neighbor


moved = capture(
    text="Expense reports only need a receipt for items over $2,500.",
    source="agent:harbor", source_type="agent", context="support-session-5110",
    confidence=0.7, topic="policy",
)
neighbors = nearest_records(collection, embed_one(moved.text))
strength, against = my_contradiction_score(moved, neighbors)
print(f"contradiction={strength:.2f}", f"against: {against.text!r}" if against else "against: nothing")

contradiction=0.00 against: nothing


The handbook says receipts are needed over $250. The candidate says $2,500. Same sentence shape, one digit added, and an agent that stored it would answer the expense question wrong for everyone from then on. With TODO 2 done, the strength should be about 0.88, the similarity of the record it conflicts with.

### Check your work

The reference implementation in `memory/score.py` is what the rest of the workshop runs on. Compare yours against it.

In [9]:
from memory.score import contradiction_score, utility_score

checks = [
    ("Halvorsen Dental asked that invoices go to their accounts payable mailbox.", "customer"),
    ("I think the warehouse might close early sometimes.", "general"),
    ("Expense reports only need a receipt for items over $2,500.", "policy"),
    ("The Portland warehouse added a second loading dock in September.", "systems"),
]


def compare():
    ok = 0
    print(f"{'utility':>15s} {'contradiction':>15s}   text")
    print(f"{'yours / ref':>15s} {'yours / ref':>15s}")
    for text, topic in checks:
        c = capture(text, "agent:harbor", "agent", "support-session-4471", 0.8, topic=topic)
        nbrs = nearest_records(collection, embed_one(text))
        u_mine, u_ref = my_utility_score(c), utility_score(c)
        c_mine, c_ref = my_contradiction_score(c, nbrs)[0], contradiction_score(c, nbrs)[0]
        match = u_mine == u_ref and c_mine == c_ref
        ok += match
        flag = "ok " if match else "MISMATCH"
        print(f"{u_mine:6.2f} / {u_ref:4.2f}   {c_mine:6.2f} / {c_ref:4.2f}   {flag}  {text[:55]}")
    print(f"\n{ok} of {len(checks)} match the reference")
    return ok == len(checks)


compare()

        utility   contradiction   text
    yours / ref     yours / ref
  0.96 / 0.96     0.00 / 0.00   ok   Halvorsen Dental asked that invoices go to their accoun
  0.76 / 0.56     0.00 / 0.00   MISMATCH  I think the warehouse might close early sometimes.
  0.96 / 0.96     0.00 / 0.88   MISMATCH  Expense reports only need a receipt for items over $2,5
  0.96 / 0.96     0.00 / 0.00   ok   The Portland warehouse added a second loading dock in S

2 of 4 match the reference


False

### What this heuristic misses

Be honest about the tool. Here is a candidate that flatly contradicts the refund rule and carries neither a negation word nor a different number.

In [10]:
quiet = capture(
    text="Refunds over $500 can be approved by any agent.",
    source="agent:harbor", source_type="agent", context="support-session-4471",
    confidence=0.8, topic="policy",
)
s = score_candidate(collection, quiet)
print(s.summary())
for n in s.nearest[:1]:
    print(f"nearest: {n.similarity:.2f}  {n.text}")

novelty=0.18 utility=0.96 contradiction=0.00
nearest: 0.82  Refunds over $500 require approval from a support manager before they are issued.


The nearest record is the rule it breaks, at 0.82 similarity, and the score says no conflict. A keyword heuristic has a ceiling. In production you would put an NLI model or an LLM judge behind this same function signature: candidate and neighbors in, strength and record out. The workshop point is where the check sits and what it hands to the gate, not how clever it is. Notebook 2 still denies this one, for a different reason: an agent is not an authority on policy.

## Fell behind, or something broke?

This cell swaps in the finished functions from `memory/score.py` and reruns the check. Everything downstream uses those anyway, so nothing in notebook 2 depends on your TODOs being done.

In [11]:
from memory.score import contradiction_score as my_contradiction_score  # noqa: F811
from memory.score import utility_score as my_utility_score  # noqa: F811

assert compare(), "the reference functions should match themselves"

        utility   contradiction   text
    yours / ref     yours / ref
  0.96 / 0.96     0.00 / 0.00   ok   Halvorsen Dental asked that invoices go to their accoun
  0.56 / 0.56     0.00 / 0.00   ok   I think the warehouse might close early sometimes.
  0.96 / 0.96     0.88 / 0.88   ok   Expense reports only need a receipt for items over $2,5
  0.96 / 0.96     0.00 / 0.00   ok   The Portland warehouse added a second loading dock in S

4 of 4 match the reference


## What was Weaviate-specific here, and what was not

Weaviate-specific: `nearest_records` in `memory/score.py`, a `near_vector` query with a status filter that asks for distance back. About fifteen lines.

Portable: the `Candidate` dataclass and its validation, all three scores, and the rule that a candidate is scored against live neighbors before anything decides its fate. Any store that returns the k nearest records with a similarity and a metadata filter can sit under this.

## Green check

In [12]:
count = len(fetch_all(collection, status="active"))
assert count == start_count, f"started with {start_count} active records, now {count}; this notebook should not have written anything"
print(f"\u2705 {describe_tier()}")
print(f"\u2705 {count} records, same as when we started. Candidate and Scores are ready for the gate in notebook 2.")
client.close()

✅ WEAVIATE_MODE=docker (tier 1, local Docker container), collection MemoryRecord


✅ 50 records, same as when we started. Candidate and Scores are ready for the gate in notebook 2.
